# Megaline Prepaid Plan Revenue Analysis

## Project Overview

In this project, I analyzed the behavior of 500 Megaline customers who used one of two prepaid plans, **Surf** or **Ultimate**, during 2018.

My goal was to understand how customers use calls, messages, and mobile data, calculate the monthly revenue generated by each plan, and use statistical hypothesis testing to determine whether the differences in revenue are significant.

I also compared customers from the NY-NJ area with customers from other regions to test whether their average monthly revenue differs.


## Objectives

In this analysis, I:

- inspected and prepared five related datasets;
- converted date fields to the appropriate data type;
- aggregated calls, messages, and internet usage by customer and month;
- calculated monthly revenue based on each plan's allowances and overage charges;
- compared customer behavior between Surf and Ultimate;
- analyzed monthly revenue distributions;
- tested whether the average revenue differs between the two plans;
- tested whether average revenue differs between NY-NJ customers and customers from other regions.


## Tools

- Python
- Pandas
- NumPy
- Matplotlib
- SciPy
- Jupyter Notebook


## Dataset

I worked with five datasets:

- `megaline_users.csv` — customer profile and plan information
- `megaline_calls.csv` — call records and call duration
- `megaline_messages.csv` — text message records
- `megaline_internet.csv` — internet session usage
- `megaline_plans.csv` — plan allowances, monthly fees, and overage prices

The notebook expects these files inside a local `data/` folder.


## 1. Import Libraries


In [ ]:
from scipy import stats as st
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt


## 2. Load the Data


In [ ]:
calls = pd.read_csv('data/megaline_calls.csv')
internet = pd.read_csv('data/megaline_internet.csv')
messages = pd.read_csv('data/megaline_messages.csv')
plans = pd.read_csv('data/megaline_plans.csv')
users = pd.read_csv('data/megaline_users.csv')


## 3. Initial Data Inspection

I first reviewed the structure, data types, missing values, and sample rows from each dataset before making any transformations.


In [ ]:
datasets = {
    'Calls': calls,
    'Internet': internet,
    'Messages': messages,
    'Plans': plans,
    'Users': users
}

for name, dataframe in datasets.items():
    print(f'\n{name}')
    print('-' * len(name))
    dataframe.info()
    display(dataframe.head())
    print('\nMissing values:')
    print(dataframe.isna().sum())


From the initial inspection, I identified that the date columns were stored as text and needed to be converted to `datetime`. The missing values in `churn_date` indicate customers who were still active when the data was collected, so I kept those values as missing rather than replacing them.


## 4. Data Preparation


In [ ]:
users['reg_date'] = pd.to_datetime(users['reg_date'])
users['churn_date'] = pd.to_datetime(users['churn_date'])

calls['call_date'] = pd.to_datetime(calls['call_date'])
messages['message_date'] = pd.to_datetime(messages['message_date'])
internet['session_date'] = pd.to_datetime(internet['session_date'])

plans['gb_per_month'] = plans['mb_per_month_included'] / 1024


### Plan Conditions

I used the plan table to calculate each customer's monthly charges.

| Plan | Monthly Fee | Included Minutes | Included Messages | Included Data |
|---|---:|---:|---:|---:|
| Surf | $20 | 500 | 50 | 15 GB |
| Ultimate | $70 | 3,000 | 1,000 | 30 GB |

Surf charges **$0.03 per extra minute**, **$0.03 per extra message**, and **$10 per extra GB**. Ultimate charges **$0.01 per extra minute**, **$0.01 per extra message**, and **$7 per extra GB**.


In [ ]:
plans


## 5. Aggregate Monthly Customer Usage

I aggregated each service by `user_id` and month so that each row would represent one customer's activity during one month.


In [ ]:
calls['month'] = calls['call_date'].dt.to_period('M')
messages['month'] = messages['message_date'].dt.to_period('M')
internet['month'] = internet['session_date'].dt.to_period('M')

calls_per_month = (
    calls.groupby(['user_id', 'month'])
         .size()
         .reset_index(name='call_count')
)

minutes_per_month = (
    calls.groupby(['user_id', 'month'])['duration']
         .sum()
         .reset_index(name='call_duration')
)

messages_per_month = (
    messages.groupby(['user_id', 'month'])
            .size()
            .reset_index(name='messages_count')
)

mb_per_month = (
    internet.groupby(['user_id', 'month'])['mb_used']
            .sum()
            .reset_index()
)


In [ ]:
merged = (
    minutes_per_month
    .merge(messages_per_month, on=['user_id', 'month'], how='outer')
    .merge(calls_per_month, on=['user_id', 'month'], how='outer')
    .merge(mb_per_month, on=['user_id', 'month'], how='outer')
)

# A missing value here means the customer did not use that service in that month.
usage_columns = ['call_duration', 'messages_count', 'call_count', 'mb_used']
merged[usage_columns] = merged[usage_columns].fillna(0)

merged = merged.merge(users, on='user_id', how='left')
merged = merged.merge(plans, left_on='plan', right_on='plan_name', how='left')

merged.head()


## 6. Calculate Monthly Revenue

I calculated each customer's monthly revenue by adding the plan's fixed monthly fee to any charges generated by usage above the included allowances.


In [ ]:
merged['extra_minutes'] = (
    merged['call_duration'] - merged['minutes_included']
).clip(lower=0)

merged['extra_messages'] = (
    merged['messages_count'] - merged['messages_included']
).clip(lower=0)

merged['gb_used'] = merged['mb_used'] / 1024

merged['extra_gb'] = (
    merged['gb_used'] - merged['gb_per_month']
).clip(lower=0)

merged['charge_minutes'] = (
    merged['extra_minutes'] * merged['usd_per_minute']
)

merged['charge_messages'] = (
    merged['extra_messages'] * merged['usd_per_message']
)

merged['charge_gb'] = (
    merged['extra_gb'] * merged['usd_per_gb']
)

merged['monthly_revenue'] = (
    merged['usd_monthly_pay']
    + merged['charge_minutes']
    + merged['charge_messages']
    + merged['charge_gb']
)

merged[['user_id', 'month', 'plan_name', 'monthly_revenue']].head()


## 7. Customer Behavior Analysis

### Calls

I compared average monthly call duration between the two plans and examined the distribution of minutes used.


In [ ]:
avg_duration = (
    merged.groupby(['plan_name', 'month'])['call_duration']
          .mean()
          .reset_index()
)

pivot_calls = avg_duration.pivot(
    index='month',
    columns='plan_name',
    values='call_duration'
)

pivot_calls.plot(kind='bar', figsize=(11, 5))
plt.title('Average Call Duration by Plan and Month')
plt.xlabel('Month')
plt.ylabel('Average Call Duration (minutes)')
plt.legend(title='Plan')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))

for plan_name in merged['plan_name'].unique():
    subset = merged.loc[
        merged['plan_name'] == plan_name,
        'call_duration'
    ]
    plt.hist(subset, bins=30, alpha=0.5, label=plan_name)

plt.axvline(500, linestyle='--', label='Surf Included Minutes')
plt.axvline(3000, linestyle='--', label='Ultimate Included Minutes')
plt.title('Monthly Call Minutes by Plan')
plt.xlabel('Minutes Used')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
call_stats_by_plan = (
    merged.groupby('plan_name')['call_duration']
          .agg(['mean', 'var', 'std', 'median'])
)

call_stats_by_plan


The average monthly call duration was very similar between the plans. In the original analysis results, Surf users averaged approximately **404.76 minutes**, while Ultimate users averaged approximately **406.19 minutes**.


### Messages


In [ ]:
message_stats = (
    merged.groupby(['plan_name', 'month'])['messages_count']
          .mean()
          .reset_index()
)

pivot_messages = message_stats.pivot(
    index='month',
    columns='plan_name',
    values='messages_count'
)

pivot_messages.plot(kind='bar', figsize=(11, 5))
plt.title('Average Number of Messages by Plan and Month')
plt.xlabel('Month')
plt.ylabel('Average Messages Sent')
plt.legend(title='Plan')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


Ultimate customers sent more messages on average in the analyzed sample. The overall averages were approximately **31.16 messages per month for Surf** and **37.55 for Ultimate**.


### Internet Usage


In [ ]:
internet_by_plan = (
    merged.groupby('plan_name')['gb_used']
          .mean()
          .reset_index()
)

plt.figure(figsize=(6, 5))
plt.bar(internet_by_plan['plan_name'], internet_by_plan['gb_used'])
plt.title('Average Internet Usage by Plan')
plt.xlabel('Plan')
plt.ylabel('Average GB Used')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

internet_by_plan


Internet consumption was also similar between the two groups. In my original results, Surf customers used approximately **16.17 GB per month**, compared with **16.81 GB for Ultimate customers**.


## 8. Revenue Analysis

I compared the distribution and average monthly revenue generated by each plan.


In [ ]:
revenue_stats = (
    merged.groupby('plan_name')['monthly_revenue']
          .agg(['count', 'mean', 'var', 'std', 'median', 'sum'])
)

revenue_stats


In [ ]:
monthly_avg_revenue = (
    merged.groupby(['month', 'plan_name'])['monthly_revenue']
          .mean()
          .reset_index()
)

pivot_revenue = monthly_avg_revenue.pivot(
    index='month',
    columns='plan_name',
    values='monthly_revenue'
)

pivot_revenue.plot(kind='bar', figsize=(11, 5))
plt.title('Average Monthly Revenue by Plan')
plt.xlabel('Month')
plt.ylabel('Average Revenue ($)')
plt.legend(title='Plan')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In my original analysis, the **Ultimate plan generated higher average monthly revenue**:

- **Surf:** approximately **$57.29** per customer-month
- **Ultimate:** approximately **$72.12** per customer-month

Surf revenue had much greater variability because customers more frequently exceeded the plan's included allowances. Ultimate had a higher base price and a much more concentrated revenue distribution.


## 9. Statistical Hypothesis Testing

### Test 1 — Surf vs. Ultimate Average Revenue

I used an independent two-sample Welch's t-test because the revenue variances between the plans were substantially different.

- **Null hypothesis (H₀):** the average monthly revenue is the same for Surf and Ultimate users.
- **Alternative hypothesis (H₁):** the average monthly revenue is different between Surf and Ultimate users.
- **Significance level:** α = 0.05


In [ ]:
surf_revenue = merged.loc[
    merged['plan_name'] == 'surf',
    'monthly_revenue'
]

ultimate_revenue = merged.loc[
    merged['plan_name'] == 'ultimate',
    'monthly_revenue'
]

alpha = 0.05

plan_test = st.ttest_ind(
    surf_revenue,
    ultimate_revenue,
    equal_var=False
)

print(f'Surf variance: {surf_revenue.var():.2f}')
print(f'Ultimate variance: {ultimate_revenue.var():.2f}')
print(f'P-value: {plan_test.pvalue:.6g}')

if plan_test.pvalue < alpha:
    print('Result: Reject H0. The average monthly revenues are significantly different.')
else:
    print('Result: Fail to reject H0. There is not enough evidence of a difference.')


The original test returned a p-value of approximately **4.88 × 10⁻²⁵**, well below 0.05. I therefore rejected the null hypothesis and concluded that the difference in average revenue between Surf and Ultimate is statistically significant.


### Test 2 — NY-NJ vs. Other Regions

I also tested whether customers in the NY-NJ area generated different average monthly revenue from customers in all other regions.

- **Null hypothesis (H₀):** NY-NJ customers and customers from other regions have the same average monthly revenue.
- **Alternative hypothesis (H₁):** their average monthly revenues are different.
- **Significance level:** α = 0.05


In [ ]:
ny_nj_revenue = merged.loc[
    merged['city'].str.contains('NY-NJ', na=False),
    'monthly_revenue'
]

other_regions_revenue = merged.loc[
    ~merged['city'].str.contains('NY-NJ', na=False),
    'monthly_revenue'
]

region_test = st.ttest_ind(
    ny_nj_revenue,
    other_regions_revenue,
    equal_var=False
)

print(f'NY-NJ variance: {ny_nj_revenue.var():.2f}')
print(f'Other regions variance: {other_regions_revenue.var():.2f}')
print(f'P-value: {region_test.pvalue:.6g}')

if region_test.pvalue < alpha:
    print('Result: Reject H0. Average monthly revenue differs between the regions.')
else:
    print('Result: Fail to reject H0. There is not enough evidence of a difference.')


The original test returned a p-value of approximately **0.0186**. Since this is below 0.05, I rejected the null hypothesis and found evidence that average monthly revenue differs between NY-NJ customers and customers from other regions.


## 10. Conclusion

From this analysis, I found that customer usage patterns were relatively similar between Surf and Ultimate for calls and internet consumption, although Ultimate customers sent somewhat more messages on average.

The largest difference appeared in revenue. **Ultimate generated the higher average monthly revenue per customer-month, approximately $72.12 compared with $57.29 for Surf**, and the statistical test confirmed that this difference was significant.

I also found that Surf customers were more likely to exceed their included allowances, which created a much wider revenue distribution. This suggests that the lower-priced Surf plan attracts heavier overage usage, while Ultimate provides more predictable revenue through its higher monthly fee and larger allowances.

Based on the average revenue objective, I would prioritize **Ultimate** when evaluating which plan generates more revenue per customer. At the same time, the overage behavior observed among Surf users suggests an opportunity to investigate whether an intermediate plan could better match the needs of customers whose usage falls between the two existing offerings.
